# G-LEAP Inference Tutorial

This notebook shows how to:
1) Set up the environment
2) Prepare or load graph dictionaries
3) Run CV ensemble inference
4) Inspect results

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shimizu-team/G-LEAP/blob/main/examples/tutorial_inference_cpu.ipynb)

## Setup

First, install the required dependencies and clone the repository (if running on Colab).

In [ ]:
# Check if running on Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone repository
    !git clone https://github.com/shimizu-team/G-LEAP.git
    %cd G-LEAP

    # Install dependencies (CPU version)
    !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
    !pip install -q torch-geometric torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.4.0+cpu.html
    !pip install -q esm rdkit unimol_tools biopython

    print("Setup complete! (CPU mode)")
else:
    print("Running locally - ensure environment is activated (CPU mode)")
    # Verify PyTorch is using CPU
    import torch
    print(f"PyTorch device available: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

## Load sample data

Each graph dict is stored as `.npy` with format:
- Protein: `{UniProt_ID.pdb: (num_nodes, features, edge_index)}`
- Ligand: `{smiles: (num_nodes, features, edge_index)}`

In [ ]:
import os
from pathlib import Path

# Set up paths
if IN_COLAB:
    repo_root = Path("/content/G-LEAP")
else:
    # Find repo root from current directory
    def find_repo_root(start: Path) -> Path:
        for path in [start] + list(start.parents):
            if (path / "src" / "inference_cv.py").exists():
                return path
        raise FileNotFoundError("Could not find repo root")
    repo_root = find_repo_root(Path.cwd())

# Change to repo root for relative paths
os.chdir(repo_root)
print(f"Working directory: {os.getcwd()}")

# Verify required files
required = ["src/inference_cv.py", "models/drug_screening",
            "examples/input_sample.csv", "data/protein_graph_sample.npy"]
for f in required:
    print(f"{f}: {'OK' if Path(f).exists() else 'MISSING'}")

In [3]:
import numpy as np

protein_graph = np.load("data/protein_graph_sample.npy", allow_pickle=True).item()
ligand_graph = np.load("data/ligand_graph_sample.npy", allow_pickle=True).item()

print(f"Proteins: {len(protein_graph)}")
print(f"Ligands:  {len(ligand_graph)}")

# Check feature dimensions
p_key, (p_n, p_feat, p_edge) = next(iter(protein_graph.items()))
l_key, (l_n, l_feat, l_edge) = next(iter(ligand_graph.items()))

print(f"Protein: {p_key}, nodes={p_n}, feat_dim={len(p_feat[0])}")
print(f"Ligand: nodes={l_n}, feat_dim={len(l_feat[0])}")

Proteins: 6
Ligands:  10
Protein: P07550.pdb, nodes=413, feat_dim=960
Ligand: nodes=30, feat_dim=881


In [4]:
import pandas as pd

df = pd.read_csv("examples/input_sample.csv")
print(f"Input samples: {len(df)}")
print(df[["UniProt ID", "smiles"]].head())

Input samples: 10
  UniProt ID                                             smiles
0     P42866  Oc1ccc2C[C@H]3N(CC4CC4)CC[C@@]45[C@@H](Oc1c24)...
1     P42866                            CC#CC#CC#C\C=C\C1OC1C=C
2     P25103  OC1CC(N(C1)C(=O)c1cn(CCCCc2nn[n-]n2)c2ccccc12)...
3     P25103                    CCCCC1CCN(CCCC(=O)c2ccccc2C)CC1
4     P21462  CC1C[C@@H](NC(=O)Nc2ccc(Cl)cc2F)C(=O)N1c1ccc(c...


## (Optional) Build your own graph dictionaries

This section demonstrates how to build graph dictionaries from raw data.
We'll download a sample PDB from AlphaFold DB and use the included sample SMILES.

**Note:** AlphaFold DB structures are available under CC-BY 4.0 license.

In [5]:
import requests
from pathlib import Path

pdb_dir = Path("data/pdb_sample")
pdb_dir.mkdir(exist_ok=True)

# GPCRs to download from AlphaFold DB
uniprot_ids = ["P42866", "P25103", "P21462", "Q92633", "P47211"]

def download_alphafold_pdb(uniprot_id, output_dir):
    """Download PDB from AlphaFold DB."""
    api_url = f"https://alphafold.ebi.ac.uk/api/prediction/{uniprot_id}"
    response = requests.get(api_url)
    if response.status_code != 200:
        raise ValueError(f"API error for {uniprot_id}")

    pdb_url = response.json()[0]["pdbUrl"]
    pdb_response = requests.get(pdb_url)

    output_path = output_dir / f"{uniprot_id}.pdb"
    output_path.write_text(pdb_response.text)
    print(f"Downloaded: {uniprot_id}")
    return output_path

# Download PDBs
for uid in uniprot_ids:
    pdb_path = pdb_dir / f"{uid}.pdb"
    if not pdb_path.exists():
        download_alphafold_pdb(uid, pdb_dir)
    else:
        print(f"Already exists: {uid}.pdb")

Already exists: P42866.pdb
Already exists: P25103.pdb
Already exists: P21462.pdb
Already exists: Q92633.pdb
Already exists: P47211.pdb


In [6]:
# Build protein graphs (CPU mode)
!python -u scripts/build_protein_graphs_esmc.py \
    --pdb_dir data/pdb_sample \
    --output_graph data/protein_graph_tutorial.npy \
    --output_embeddings data/protein_embeddings_tutorial.pt \
    --device cpu

Found 6 PDB files.
Loading ESMC model...
Fetching 4 files: 100%|████████████████████████| 4/4 [00:00<00:00, 78033.56it/s]
/gs/bs/tga-asm/tsaku/.conda/envs/gleap/lib/python3.10/site-packages/esm/pretrained.py:74: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. P

In [7]:
# Build ligand graphs (CPU mode - disable CUDA)
import os
os.environ['CUDA_VISIBLE_DEVICES'] = ''  # Hide GPU from the script

!python -u scripts/build_compound_graphs_unimol.py \
    --csv_file examples/ligands_sample.csv \
    --output_graph data/ligand_graph_tutorial.npy

Initializing Uni-Mol model...
2026-02-05 16:38:51 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /gs/bs/tga-asm/tsaku/.conda/envs/gleap/lib/python3.10/site-packages/unimol_tools/weights/modelzoo/164M/checkpoint.pt
2026-02-05 16:38:51 | unimol_tools/models/unimolv2.py | 176 | INFO | Uni-Mol Tools | Loading pretrained weights from /gs/bs/tga-asm/tsaku/.conda/envs/gleap/lib/python3.10/site-packages/unimol_tools/weights/modelzoo/164M/checkpoint.pt
Uni-Mol model initialized (unimolv2, 164m)
Processing compounds:   0%|                              | 0/10 [00:00<?, ?it/s]2026-02-05 16:38:51 | unimol_tools/data/conformer.py | 452 | INFO | Uni-Mol Tools | Start generating conformers...
Uni-Mol model initialized (unimolv2, 164m)
Processing compounds:   0%|                              | 0/10 [00:00<?, ?it/s]2026-02-05 16:38:51 | unimol_tools/data/conformer.py | 452 | INFO | Uni-Mol Tools | Start generating conformers...

0it [00:00, ?it/s]
0it [0

## Run CV ensemble inference (sample data)

This uses the published 10-fold ensemble with the sample graphs.

In [8]:
# Run inference with 10-fold CV ensemble (CPU mode)
!python -u src/inference_cv.py \
    --model_dir models/drug_screening \
    --csv_file examples/input_sample.csv \
    --protein_graph data/protein_graph_sample.npy \
    --ligand_graph data/ligand_graph_sample.npy \
    --output_file examples/output.csv \
    --device cpu

=== GPCR-Ligand Interaction Prediction CV Inference ===
Model directory: models/drug_screening
Input CSV: examples/input_sample.csv
Output file: examples/output.csv
Model type: gcn_bilinear
Split type: protein
N folds: 10
Ensemble method: mean
Using device: cpu
Loading model for fold 0: models/drug_screening/pretrained_model_fold0.pt
Loaded fold 0 model successfully
Loading model for fold 1: models/drug_screening/pretrained_model_fold1.pt
Loaded fold 0 model successfully
Loading model for fold 1: models/drug_screening/pretrained_model_fold1.pt
Loaded fold 1 model successfully
Loading model for fold 2: models/drug_screening/pretrained_model_fold2.pt
Loaded fold 1 model successfully
Loading model for fold 2: models/drug_screening/pretrained_model_fold2.pt
Loaded fold 2 model successfully
Loading model for fold 3: models/drug_screening/pretrained_model_fold3.pt
Loaded fold 2 model successfully
Loading model for fold 3: models/drug_screening/pretrained_model_fold3.pt
Loaded fold 3 model su

In [9]:
# View results
df_out = pd.read_csv("examples/output.csv")
print(f"Predictions: {len(df_out)}")
print(f"Positive (score > 0.5): {sum(df_out['CV_Ensemble_Binary'])}")
df_out[["UniProt_ID", "SMILES", "CV_Ensemble_Score", "CV_Ensemble_Binary", "Label"]].head(10)

Predictions: 10
Positive (score > 0.5): 5


,UniProt_ID,SMILES,CV_Ensemble_Score,CV_Ensemble_Binary,Label
0,P42866,Oc1ccc2C[C@H]3N(CC4CC4)CC[C@@]45[C@@H](Oc1c24)...,0.974906,1,1.0
1,P42866,CC#CC#CC#C\C=C\C1OC1C=C,0.164329,0,0.0
2,P25103,OC1CC(N(C1)C(=O)c1cn(CCCCc2nn[n-]n2)c2ccccc12)...,0.983202,1,1.0
3,P25103,CCCCC1CCN(CCCC(=O)c2ccccc2C)CC1,0.022298,0,0.0
4,P21462,CC1C[C@@H](NC(=O)Nc2ccc(Cl)cc2F)C(=O)N1c1ccc(c...,0.585338,1,1.0
5,P21462,CSc1ccccc1NS(=O)(=O)c1ccccc1,0.004605,0,0.0
6,Q92633,CCCCCCCCCCCCCCCCCCOC[C@H](COP(O)(O)=S)OC,0.773180,1,1.0
7,Q92633,C[C@@H](OC(=O)Nc1c(cnn1C)-c1ccc(CC(O)=O)cc1)c1...,0.006549,0,0.0
8,P47211,COc1ccccc1N(CCOc1ccc(C)cc1)C(=O)CN1CCCCC1,0.609974,1,1.0
9,P47211,COc1ccc(Nc2cc(nc(n2)N2CCCC2)N2CCN(CC2)c2ccccc2...,0.010141,0,0.0


## Metabolite screening (optional)

In [10]:
# Metabolite screening (CPU mode)
!python -u src/inference_cv.py \
    --model_dir models/metabolite_screening \
    --csv_file examples/metabolite_input_sample.csv \
    --protein_graph data/metabolite_protein_graph_sample.npy \
    --ligand_graph data/metabolite_ligand_graph_sample.npy \
    --output_file examples/metabolite_output.csv \
    --device cpu

=== GPCR-Ligand Interaction Prediction CV Inference ===
Model directory: models/metabolite_screening
Input CSV: examples/metabolite_input_sample.csv
Output file: examples/metabolite_output.csv
Model type: gcn_bilinear
Split type: protein
N folds: 10
Ensemble method: mean
Using device: cpu
Loading model for fold 0: models/metabolite_screening/pretrained_model_fold0.pt
Loaded fold 0 model successfully
Loading model for fold 1: models/metabolite_screening/pretrained_model_fold1.pt
Loaded fold 0 model successfully
Loading model for fold 1: models/metabolite_screening/pretrained_model_fold1.pt
Loaded fold 1 model successfully
Loading model for fold 2: models/metabolite_screening/pretrained_model_fold2.pt
Loaded fold 1 model successfully
Loading model for fold 2: models/metabolite_screening/pretrained_model_fold2.pt
Loaded fold 2 model successfully
Loading model for fold 3: models/metabolite_screening/pretrained_model_fold3.pt
Loaded fold 2 model successfully
Loading model for fold 3: models